In [ ]:
#Categorical Features

from pyspark.ml.feature import StringIndexer

failure_indexer = StringIndexer(inputCol="failure_type", outputCol="failure_type_index")
df_features = failure_indexer.fit(df_silver).transform(df_silver)


In [ ]:
#Time Series

from pyspark.sql.functions import monotonically_increasing_id

df_features = df_features.withColumn("row_id", monotonically_increasing_id())


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, lag

window_spec = Window.orderBy("row_id")

# Rolling average of torque
df_features = df_features.withColumn("avg_torque_last3", avg("torque_nm").over(window_spec.rowsBetween(-2, 0)))

# Lag feature: previous tool wear
df_features = df_features.withColumn("prev_tool_wear", lag("tool_wear_min", 1).over(window_spec))


In [ ]:
# Save as Parquet to simplify schema + speed up reading
silver_output_path = "/content/silver_output.parquet"

df_silver.write.mode("overwrite").parquet(silver_output_path)
print("✅ Saved silver layer as Parquet.")


In [ ]:
import shutil

# Compress the Parquet file
shutil.make_archive("/content/silver_output", 'zip', "/content", "silver_output.parquet")
files.download("/content/silver_output.zip")
